# Fine-tune GR00T-N1.7 with NVIDIA's `launch_finetune.py`

Vendor path: dataset loading, video decode, normalization, the HF `Trainer` loop,
checkpointing and wandb are all NVIDIA's code. This notebook only supplies the
config -- every hyperparameter and path comes from `configs/finetune_vendor.yaml`
through `training_cfgs.Config` -- stages the dataset, launches the script, and plots the loss.

Companion to `finetune_scratch.ipynb`, which builds the model and training loop by hand.

Run in `$HPCWORK/gr00t-ft/venvs/gr00t` with `module load FFmpeg/7.1.2` loaded, on a GPU.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from huggingface_hub import snapshot_download
from training_cfgs import Config

REPO_ROOT = Path.home() / "finetune-groot"
VENDOR = REPO_ROOT / "vendor" / "Isaac-GR00T"
LAUNCH_SCRIPT = VENDOR / "gr00t" / "experiment" / "launch_finetune.py"

cfg = Config.from_file(REPO_ROOT / "configs" / "finetune_vendor.yaml")
# cfg.apply_args({"training.max_steps": 10000})  # dotted overrides, e.g. for a full run
print(cfg)

## Stage the dataset

The Hub repo is the plain LeRobot v2.1 tree. The vendor script also needs three
files that are not on the Hub: `meta/modality.json` (column slices, copied from
`configs/modality.json`) and `meta/stats.json` + `meta/relative_stats.json`
(generated by the vendor's `stats.py` against the embodiment config). Safe to
re-run: already-downloaded files are skipped and `stats.py` recomputes only stats
whose fingerprint no longer matches. Delete the two stats files to force a rebuild
after changing `embodiment_config.py`.

In [ ]:
dataset = Path(cfg.training.dataset_path)

snapshot_download(cfg.data.repo_id, repo_type="dataset", revision=cfg.data.revision, local_dir=dataset)
shutil.copy(cfg.data.modality_json, dataset / "meta" / "modality.json")
subprocess.run([sys.executable, str(VENDOR / "gr00t" / "data" / "stats.py"),
                "--dataset-path", str(dataset),
                "--embodiment-tag", cfg.model.embodiment_tag,
                "--modality-config-path", cfg.model.modality_config_path], check=True)

print(sorted(p.name for p in (dataset / "meta").iterdir()))

## Launch

`cfg.flatten()` turns the `model` and `training` groups into the flat flags
`launch_finetune.py` expects (`--tune-projector`, `--no-tune-llm`, `--max-steps 200`, ...).
Output streams below.

In [ ]:
cmd = [sys.executable, str(LAUNCH_SCRIPT), *cfg.flatten(groups=["model", "training"])]
print(" ".join(cmd), "\n")

proc = subprocess.Popen(cmd, cwd=REPO_ROOT, text=True, bufsize=1,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "launch_finetune.py failed"

## Loss

The run is written to `<output_dir>/<experiment_name>/`; the latest
`checkpoint-*/trainer_state.json` holds the full `log_history`.

In [ ]:
run_dir = Path(cfg.training.output_dir) / cfg.training.experiment_name
latest = max(run_dir.glob("checkpoint-*/trainer_state.json"), key=lambda p: int(p.parent.name.split("-")[1]))
log = [r for r in json.loads(latest.read_text())["log_history"] if "loss" in r]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([r["step"] for r in log], [r["loss"] for r in log], marker="o", ms=3)
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.set_title(run_dir.name)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()